# Powder XRD Analyzer - Interactive Workflow

This notebook demonstrates the complete workflow for analyzing Powder XRD data from single crystal slab samples.

## Steps:
1. Load crystal structure from CIF file
2. Calculate theoretical powder pattern
3. Load and preprocess experimental XRD data
4. Find peaks in experimental data
5. Match and index peaks (determine hkl)
6. Analyze slab orientation

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, '.')

import matplotlib.pyplot as plt

---
## Configuration

Update these paths to point to your actual data files.

In [ ]:
# Configuration - update these paths!
CIF_FILE = "your_structure.cif"
XRD_DATA = "your_xrd_data.txt"

# Parameters
WAVELENGTH = "CuKa"
MIN_2THETA = 5
MAX_2THETA = 90
PEAK_HEIGHT_PCT = 1.0
PEAK_PROMINENCE = 5.0
MATCH_TOLERANCE = 0.1
ORIENTATION_MAX_INDEX = 3

---
## 1. Load Crystal Structure from CIF File

In [ ]:
from powder_xrd_analyzer.io import read_cif

try:
    structure = read_cif(CIF_FILE)
    print(f"Successfully loaded: {structure.composition.reduced_formula}")
    print(f"Space group: {structure.get_space_group_info()[0]}")
    print(f"Lattice parameters: {structure.lattice}")
except FileNotFoundError:
    print(f"Note: {CIF_FILE} not found. This is an example workflow!")
    print("Replace CIF_FILE with your actual CIF file path.")
    print("\nProceeding with demonstration...")
    structure = None

---
## 2. Calculate Theoretical Powder Pattern

In [ ]:
if structure:
    from powder_xrd_analyzer.pattern_calculator import calculate_powder_pattern, print_pattern_summary

    pattern = calculate_powder_pattern(
        structure,
        wavelength=WAVELENGTH,
        two_theta_range=(MIN_2THETA, MAX_2THETA)
    )

    print(f"Calculated {len(pattern.x)} peaks:")
    print("=" * 70)
    print_pattern_summary(pattern, n_peaks=20)
else:
    print("No structure loaded - skipping pattern calculation.")
    pattern = None

In [ ]:
if pattern:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.vlines(pattern.x, 0, pattern.y, 'r', linewidth=2)
    ax.set_xlabel('2θ (degrees)')
    ax.set_ylabel('Relative Intensity')
    ax.set_title('Calculated Powder Pattern')
    ax.grid(alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.show()
else:
    print('No pattern to plot.')

---
## 3. Load Experimental XRD Data

In [ ]:
from powder_xrd_analyzer.io import auto_read_xrd

try:
    tt, intensity = auto_read_xrd(XRD_DATA)
    print(f"Loaded {len(tt)} data points")
    print(f"2θ range: {tt[0]:.2f}° - {tt[-1]:.2f}°")
    print(f"Intensity range: {intensity.min():.1f} - {intensity.max():.1f}")
except FileNotFoundError:
    print(f"Note: {XRD_DATA} not found. Creating simulated data for demo...")
    
    # Create simulated data if no real data is available
    import numpy as np
    tt = np.linspace(5, 80, 3000)
    intensity = np.random.randn(len(tt)) * 10 + 50
    
    if pattern:
        for peak_tt, peak_int in zip(pattern.x[:10], pattern.y[:10]):
            idx = np.argmin(np.abs(tt - peak_tt))
            intensity[idx-10:idx+10] += peak_int * 10
    
    print(f"Simulated data: {len(tt)} points")

In [ ]:
# Plot raw experimental data
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(tt, intensity, 'b-', linewidth=0.6)
ax.set_xlabel('2θ (degrees)')
ax.set_ylabel('Intensity')
ax.set_title('Raw Experimental XRD Data')
ax.grid(alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

---
## 4. Preprocess and Find Peaks

We apply:
- Smoothing (Savitzky-Golay filter)
- Background subtraction (SNIP algorithm)

In [ ]:
from powder_xrd_analyzer.peak_finding import preprocess_xrd_data, find_xrd_peaks

# Preprocess
processed = preprocess_xrd_data(tt, intensity, smooth=True, bg_method='snip')

# Find peaks
peak_tt, peak_int, peak_idx = find_xrd_peaks(
    tt,
    processed,
    height_pct=PEAK_HEIGHT_PCT,
    min_distance=5,
    prominence=PEAK_PROMINENCE
)

print(f"Found {len(peak_tt)} peaks:")
print("=" * 40)
print(f"{'Peak':>5}  {'2θ (°)':>10}  {'Intensity':>12}")
print("-" * 40)
for i, (p_tt, p_int) in enumerate(zip(peak_tt, peak_int)):
    print(f"{i+1:>5}  {p_tt:>10.3f}  {p_int:>12.2f}")

In [ ]:
# Plot detected peaks
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(tt, intensity, 'b-', linewidth=0.6, label='Raw data')
ax1.set_ylabel('Intensity')
ax1.set_title('Raw Data')
ax1.legend()
ax1.grid(alpha=0.3, linestyle='--')

ax2.plot(tt, processed, 'g-', linewidth=0.6, label='Processed')
ax2.plot(peak_tt, peak_int, 'ro', markersize=6, label='Peaks', zorder=5)
ax2.set_xlabel('2θ (degrees)')
ax2.set_ylabel('Intensity')
ax2.set_title(f'Processed Data with {len(peak_tt)} Detected Peaks')
ax2.legend()
ax2.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

---
## 5. Match and Index Peaks (hkl determination)

Match experimental peaks with the calculated pattern to determine Miller indices.

In [ ]:
if pattern:
    from powder_xrd_analyzer.peak_matching import match_peaks, print_indexed_peaks

    matches = match_peaks(peak_tt, pattern, tolerance=MATCH_TOLERANCE)
    
    print(f"Matched {len(matches)} of {len(peak_tt)} peaks:")
    print("=" * 80)
    print_indexed_peaks(matches, show_unmatched=True, observed_peaks=peak_tt)
else:
    print('No calculated pattern available for matching.')
    matches = []

In [ ]:
if pattern and matches:
    from powder_xrd_analyzer.visualization import plot_comparison

    fig = plot_comparison(tt, processed, pattern, matches)
    plt.show()

In [ ]:
# Show unique Miller indices found
if matches:
    from powder_xrd_analyzer.peak_matching import get_unique_miller_indices
    
    hkls = get_unique_miller_indices(matches)
    print(f"Found {len(hkls)} unique Miller indices:")
    for hkl in hkls:
        print(f"  ({hkl[0]}{hkl[1]}{hkl[2]})")
else:
    print('No matches found.')

---
## 6. Analyze Slab Orientation

Generate symmetrically-distinct slab orientations and match peak patterns to determine the most likely orientation of your single crystal slab.

In [ ]:
if structure and len(peak_tt) > 0:
    from powder_xrd_analyzer.orientation import analyze_slab_orientation, print_orientation_results

    print(f"Analyzing orientations up to Miller index {ORIENTATION_MAX_INDEX}...")
    print('This may take a minute...')
    
    results = analyze_slab_orientation(
        structure,
        peak_tt,
        max_index=ORIENTATION_MAX_INDEX,
        tolerance=MATCH_TOLERANCE * 1.5,
        wavelength=WAVELENGTH,
        two_theta_range=(MIN_2THETA, MAX_2THETA)
    )
    
    print('\n' + '=' * 60)
    print('Top Orientation Candidates')
    print('=' * 60)
    print_orientation_results(results, n_top=5)
else:
    print('Skipping orientation analysis (need structure and peaks).')
    results = []

In [ ]:
# Summary of best orientation
if results:
    best = results[0]
    print('=' * 60)
    print('BEST ORIENTATION CANDIDATE')
    print('=' * 60)
    print(f'Miller index: {best["miller_index"]}')
    print(f'Match ratio: {best["match_ratio"]:.1%}')
    print(f'Matched peaks: {best["n_matched"]}/{best["total_peaks"]}')
    print(f'Matched peak positions (2θ): {[f"{p:.2f}°" for p in best["matched_peaks"]]}')

---
## Summary

You have successfully:
1. ✅ Loaded a crystal structure from CIF file
2. ✅ Calculated a theoretical powder pattern
3. ✅ Loaded and preprocessed experimental XRD data
4. ✅ Detected peaks in the experimental data
5. ✅ Matched peaks and determined Miller indices (hkl)
6. ✅ Analyzed single crystal slab orientation

### Next Steps
- Adjust `PEAK_HEIGHT_PCT` and `PEAK_PROMINENCE` if too many/few peaks are detected
- Adjust `MATCH_TOLERANCE` if peaks are not being matched correctly
- Increase `ORIENTATION_MAX_INDEX` to consider higher-index orientations
- Use the command line scripts for batch processing:
  ```bash
  python scripts/index_peaks.py --cif your.cif --data your_data.txt
  python scripts/analyze_orientation.py --cif your.cif --data your_data.txt
  ```

In [ ]:
print('Workflow complete!')